# Device Lookup Batch — Bronze → Silver Cleaning

Cleans the `device_lookup_batch` bronze data and publishes the Silver table.

**Pipeline**
1. **Data Cleaning** — parse `notes`, standardize timestamps/dates, project to the 20-column Silver schema, dedupe, and stage to Parquet.
2. **Data Validation** — reload the staged Parquet and run business-contract checks (FAIL counts double as data-quality KPIs for carrier reporting).
3. **Data Checking** — drill into specific data-quality problems with bronze-side views.
4. **Save Data** — publish to the partitioned Iceberg Silver table.

**Input:** pre-decompressed bronze at `s3://enstream-lake-silver-dev/temp_raw/device_lookup_batch/`  
**Output:** staged Parquet + Iceberg table `trust_score_v1_poc_silver.device_lookup_batch`

Unit / invariant tests for the cleaning functions live in the companion notebook `device_lookup_batch_tests_v6.ipynb`.

In [ ]:
%%pyspark project.spark.compatibility
import boto3

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType

Executing for connection type: SPARK_GLUE, connection name: project.spark.compatibility


In [2]:
%%pyspark project.spark.compatibility
spark = (
    SparkSession.builder
    .appName("device_lookup_batch")
    .config("spark.executor.memory", "18g")
    .config("spark.driver.memory", "10g")
    .config("spark.executor.cores", "4")
    .config("spark.sql.shuffle.partitions", "300")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)


Connection: project.spark.compatibility | Run start time: 2026-08-14 20:17:10.148431 | Run duration : 0:00:00.738012s.


In [3]:
%%pyspark project.spark.compatibility
# Bronze `timestamp` is already in America/Toronto wall-clock time (EST/EDT).
# Set session tz to match so to_timestamp preserves the wall-clock string on parse.
spark.conf.set("spark.sql.session.timeZone", "America/Toronto")



Connection: project.spark.compatibility | Run start time: 2026-08-14 20:17:10.889459 | Run duration : 0:00:02.659603s.


In [4]:
%%pyspark project.spark.compatibility
# -------------------------------------------------------------------------
# Original bronze read (kept for reference; very slow due to gzip on read).
# One-time prep was: decompress + write to temp_raw parquet (cell below).
# -------------------------------------------------------------------------
# s3_path = "s3://enstream-lake-bronze-dev/landing/device_lookup_batch.csv.gz"
# dlb = spark.read.option("header", True).csv(s3_path)
# dlb.printSchema()
# dlb.write.mode("overwrite").parquet("s3://enstream-lake-silver-dev/temp_raw/device_lookup_batch/")

Connection: project.spark.compatibility | Run start time: 2026-08-02 01:22:09.649249 | Run duration : 0:00:00.030892s.


In [4]:
%%pyspark project.spark.compatibility
# Read the pre-decompressed bronze copy (much faster than re-reading .csv.gz)
dlb = spark.read.parquet("s3://enstream-lake-silver-dev/temp_raw/device_lookup_batch/")
dlb.printSchema()
dlb.show(20, truncate=False)

root
 |-- id: string (nullable = true)
 |-- change_key: string (nullable = true)
 |-- change_key_ats: string (nullable = true)
 |-- mno: string (nullable = true)
 |-- imei: string (nullable = true)
 |-- imsi: string (nullable = true)
 |-- date: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- notes: string (nullable = true)
 |-- related_id: string (nullable = true)
 |-- status: string (nullable = true)

+---+----------------------------------------------------------------+--------------+-----+----------------------------------------------------------------+----------------------------------------------------------------+----------+-------------------+------------------+----------+----------+
|id |change_key                                                      |change_key_ats|mno  |imei                                                            |imsi                                                            |date      |timestamp          |notes             |relate

In [7]:
%%pyspark project.spark.compatibility
imei_value = "e486bcbb243eee8089e634703f96083b49c36b37cbf7d72148e906387ea0d23d"

dlb_check = spark.read.parquet("s3://enstream-lake-silver-dev/temp_raw/device_lookup_batch/")

matched_parquet = dlb_check.filter(F.trim(F.col("imei")) == imei_value)
print(f"temp_raw parquet match count: {matched_parquet.count()}")
matched_parquet.show(truncate=False, vertical=True)


temp_raw parquet match count: 1
-RECORD 0---------------------------------------------------------------------------------------------------------
 id             | 3406498052                                                                                      
 change_key     | 98dd7bb82db340ec818a5bcc1e24a42e5f27ed0d61bc08500fde1a02c5b35ae1                                
 change_key_ats | 14a3738c1616aecfc0b3ad5d81af0074b252e48bb74fa50358189202cad75038                                
 mno            | BELL                                                                                            
 imei           | e486bcbb243eee8089e634703f96083b49c36b37cbf7d72148e906387ea0d23d                                
 imsi           | NULL                                                                                            
 date           | 2025-10-12                                                                                      
 timestamp      | 2025-10-12 15:18:58           

In [7]:
%%pyspark project.spark.compatibility
spark.table("trust_score_v1_poc_silver.device_lookup_batch").show(20, False)

+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+------+----------------------------------------------------------------+----------------------------------------------------------------+----------+-------------------+----------------------------------------+----------+------------+--------+-------+----------------------------------------------------------------+---------------+-------------------+-----------+---------------+--------------+----------+
|record_id                                                       |phone_number_AC_hash                                            |phone_number_AT_hash                                            |mno   |imei                                                            |imsi                                                            |date      |event_timestamp    |correlation_id                    

In [6]:
%%pyspark project.spark.compatibility
# Dedup fully-identical bronze rows UP FRONT, before any transform.
# Required by the new record_id: it is now a per-row sha256 surrogate, so an
# end-of-pipeline dropDuplicates() would remove nothing (every row's hash differs).
# Deduping here also saves compute (duplicates never hit flatten / standardize).
dlb = dlb.dropDuplicates()
dlb.cache()
dlb.count()



412347877
Connection: project.spark.compatibility | Run start time: 2026-07-09 02:16:41.971181 | Run duration : 0:09:21.978910s.


# Data Cleaning

Parse the `notes` JSON, standardize timestamps/dates, project to the 20-column Silver schema, dedupe, and stage to Parquet.

In [7]:
%%pyspark project.spark.compatibility
def profile_json_keys(
    df: DataFrame,
    json_col: str,
    treat_backslash_n_as_null: bool = True,
    empty_as_null: bool = True,
) -> DataFrame:
    """List all keys appearing in a JSON-string column with their occurrence counts."""
    condition = F.col(json_col).isNotNull()
    if treat_backslash_n_as_null:
        condition = condition & (F.col(json_col) != "\\N")
    if empty_as_null:
        condition = condition & (F.trim(F.col(json_col)) != "")

    return (
        df.select(json_col)
          .filter(condition)
          .withColumn("json_map", F.from_json(F.col(json_col), "map<string,string>"))
          .select(F.explode(F.map_keys(F.col("json_map"))).alias("json_key"))
          .groupBy("json_key").count()
          .orderBy(F.desc("count"))
    )


Connection: project.spark.compatibility | Run start time: 2026-07-09 02:26:03.955384 | Run duration : 0:00:00.765180s.


In [8]:
%%pyspark project.spark.compatibility
profile_json_keys(dlb, "notes")

Connection: project.spark.compatibility | Run start time: 2026-07-09 02:26:04.724875 | Run duration : 0:08:46.054337s.


In [9]:
%%pyspark project.spark.compatibility
def flatten_json_column_with_schema(
    df: DataFrame,
    json_col: str,
    schema: StructType,
    prefix: str = None,
    drop_original: bool = False,
) -> DataFrame:
    """Parse a JSON-string column with a given schema and promote each field to its own column."""
    parsed = F.from_json(F.col(json_col), schema)
    for field in schema.fieldNames():
        output_col = f"{prefix}_{field}" if prefix else field
        df = df.withColumn(output_col, parsed[field])
    if drop_original:
        df = df.drop(json_col)
    return df


Connection: project.spark.compatibility | Run start time: 2026-07-09 02:34:51.177245 | Run duration : 0:00:00.751205s.


In [10]:
%%pyspark project.spark.compatibility
# notes JSON schema for device_lookup_batch.
# correlationId may not appear in every row; will be null where absent.
notes_schema = StructType([
    StructField("tac",             StringType(), True),
    StructField("oldIMSI",         StringType(), True),
    StructField("oldIMEI",         StringType(), True),
    StructField("fromADCSnapshot", StringType(), True),
    StructField("correlationId",   StringType(), True),
])


Connection: project.spark.compatibility | Run start time: 2026-07-09 02:34:51.932341 | Run duration : 0:00:00.747399s.


In [11]:
%%pyspark project.spark.compatibility
# notes schema drift guard — fail loudly if upstream adds a key we don't parse,
# instead of silently dropping it. Keep EXPECTED_NOTES_KEYS in sync with notes_schema.
EXPECTED_NOTES_KEYS = {"tac", "oldIMSI", "oldIMEI", "fromADCSnapshot", "correlationId"}

observed = {r["json_key"] for r in profile_json_keys(dlb, "notes").collect()}
unexpected = observed - EXPECTED_NOTES_KEYS
if unexpected:
    raise ValueError(
        f"notes schema drift: unexpected keys {sorted(unexpected)} — update notes_schema before rerun"
    )



Connection: project.spark.compatibility | Run start time: 2026-07-09 02:34:52.684103 | Run duration : 0:01:50.203075s.


In [12]:
%%pyspark project.spark.compatibility
dlb_parsed = flatten_json_column_with_schema(
    df=dlb,
    json_col="notes",
    schema=notes_schema,
    prefix=None,
    drop_original=True,
)


Connection: project.spark.compatibility | Run start time: 2026-07-09 02:36:42.891469 | Run duration : 0:00:00.732438s.


In [13]:
%%pyspark project.spark.compatibility
def is_blank(c):
    """Return True when `c` is null, "\\N", empty, whitespace-only, or "MISSING_DATA"."""
    t = F.trim(F.col(c))
    return F.col(c).isNull() | (t == "\\N") | (t == "") | (t == "MISSING_DATA")


Connection: project.spark.compatibility | Run start time: 2026-07-09 02:36:43.627925 | Run duration : 0:00:00.259846s.


In [14]:
%%pyspark project.spark.compatibility
def standardize_timestamp(df, ts_col, output_col=None):
    """Parse a string timestamp (yyyy-MM-dd HH:mm:ss or yyyy-MM-dd) into TimestampType; unparseable -> null. Wall-clock time is interpreted in the session timezone (set to America/Toronto)."""
    target_col = output_col or ts_col
    parsed = F.coalesce(
        F.try_to_timestamp(F.col(ts_col), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.col(ts_col), F.lit("yyyy-MM-dd")),
    )
    return df.withColumn(target_col, parsed)

def standardize_date(df, date_col, output_col=None):
    """Parse a string date column (yyyy-MM-dd) into DateType; unparseable -> null.
    Pre-filters to the yyyy-MM-dd pattern before to_date so bad input becomes null
    under any Spark version (no try_to_date dependency)."""
    target_col = output_col or date_col
    s = F.col(date_col)
    safe = F.when(s.rlike(r"^\d{4}-\d{2}-\d{2}$"), s).otherwise(None)
    return df.withColumn(target_col, F.to_date(safe, "yyyy-MM-dd"))


Connection: project.spark.compatibility | Run start time: 2026-07-09 02:36:43.891459 | Run duration : 0:00:00.743407s.


In [15]:
%%pyspark project.spark.compatibility
dlb_ts    = standardize_timestamp(dlb_parsed, "timestamp")
dlb_dated = standardize_date(dlb_ts, "date")


Connection: project.spark.compatibility | Run start time: 2026-07-09 02:36:44.638910 | Run duration : 0:00:00.319099s.


In [16]:
%%pyspark project.spark.compatibility
# Project bronze -> silver schema (20 columns)

def clean_str(c):
    """Trim; convert literal "\\N", empty string, and "MISSING_DATA"
    (Rogers placeholder for unknown values) to null."""
    v = F.trim(F.col(c))
    return F.when((v == "\\N") | (v == "") | (v == "MISSING_DATA"), None).otherwise(v)

def safe_long(c):
    """Cast a string column to bigint. Pure digits only; non-numeric -> null.
    LongType required: bronze `id` is bigint and values can exceed Int32 max."""
    s = clean_str(c)
    return F.when(s.rlike("^[0-9]+$"), s).otherwise(None).cast(LongType())

def to_silver(df):
    """Project the dlb frame onto the silver schema (20 columns).
    `event_date` is derived from `event_timestamp` so the partition column is
    always internally consistent with the event time; bronze `date` is kept
    alongside for source-fidelity tracking.

    Key / id handling:
      - record_id       = sha2(id) surrogate key, decoupled from bronze id.
                          bronze id is verified unique + non-null, so a single-column
                          hash is collision-free. STRING (64-hex), deterministic ->
                          stable across reruns (safe for Iceberg overwritePartitions).
      - source_event_id = bronze id (the source-system primary key).
      - correlation_id  = real value if present, else a deterministic synthetic
                          "gen-" + 36-char UUID derived from id. device_lookup notes
                          carry no real correlationId, so this fills every row; the
                          "gen-" prefix keeps synthetic rows filterable
                          (WHERE correlation_id LIKE 'gen-%').
    """
    # Deterministic 36-char UUID shape from the unique, non-null id.
    # Used only to synthesize correlation_id where no real value exists.
    _h = F.sha2(clean_str("id"), 256)                                        # 64 hex
    _uuid_from_id = F.concat_ws("-",
        F.substring(_h, 1, 8),  F.substring(_h, 9, 4),  F.substring(_h, 13, 4),
        F.substring(_h, 17, 4), F.substring(_h, 21, 12),
    )                                                                        # 36-char UUID shape

    return df.select(
        F.sha2(clean_str("id"), 256).alias("record_id"),                     # #3 surrogate (was safe_long id)
        clean_str("change_key").alias("phone_number_AC_hash"),
        clean_str("change_key_ats").alias("phone_number_AT_hash"),
        F.upper(clean_str("mno")).alias("mno"),
        clean_str("imei").alias("imei"),
        clean_str("imsi").alias("imsi"),
        F.col("date"),                                                       # bronze date kept
        F.col("timestamp").alias("event_timestamp"),
        F.coalesce(                                                          # #5 real value, else gen- synthetic
            F.lower(clean_str("correlationId")),
            F.concat(F.lit("gen-"), _uuid_from_id),
        ).alias("correlation_id"),
        clean_str("related_id").alias("related_id"),                         # #7 unchanged, pending Simon
        F.upper(clean_str("status")).alias("event_type"),
        clean_str("tac").alias("tac"),
        clean_str("oldIMSI").alias("oldIMSI"),
        clean_str("oldIMEI").alias("oldIMEI"),
        F.lower(clean_str("fromADCSnapshot")).alias("fromADCSnapshot"),      # #6 standardize (lower); null preserved
        F.to_timestamp(F.lit("2026-03-16 00:00:00")).alias("ingestion_ts"),
        F.lit("MYSQL").alias("source_name"),
        safe_long("id").alias("source_event_id"),                            # #2 bronze id (was null)
        F.lit(1).cast(IntegerType()).alias("schema_version"),
        F.to_date(F.col("timestamp")).alias("event_date"),                   # derived, partition col
    )

dlb_final = to_silver(dlb_dated)



Connection: project.spark.compatibility | Run start time: 2026-07-09 02:36:44.962245 | Run duration : 0:00:02.357216s.


In [17]:
%%pyspark project.spark.compatibility
# Dedup moved to the top of the pipeline (right after the bronze read). record_id is
# now a per-row sha256 surrogate, so an end-of-pipeline dropDuplicates() would remove
# nothing here (every row's record_id differs). Kept as a marker; no-op by design.


Connection: project.spark.compatibility | Run start time: 2026-07-09 02:36:47.323528 | Run duration : 0:00:00.043698s.


In [18]:
%%pyspark project.spark.compatibility
dlb_final.show(1, False, True)

-RECORD 0--------------------------------------------------------------------------------
 record_id            | 00f4b15ad27104354255e3e58284b6079d87d78db0e113a31bbb511c4ff5fea2 
 phone_number_AC_hash | 9fd50c0d2a2f6a0d9b29288eb7030e5b4ad35c2372941ad9ea0ffbd3832a096d 
 phone_number_AT_hash | 3bd7e19b605bf700f694038408fc1c9e307a95a29ccfa71e884f7462bce9e9f2 
 mno                  | TELUS                                                            
 imei                 | 3eea4629a23bb8178bda13eeaed1b6dddd02fa8e51cb1cff9affb7d6b19f70f3 
 imsi                 | 4c6cadcb41c1073bdb5be4d5e352f5ddaaac639bb71ebef03c37c94c60341230 
 date                 | 2024-01-16                                                       
 event_timestamp      | 2024-01-16 14:42:25                                              
 correlation_id       | gen-00f4b15a-d271-0435-4255-e3e58284b607                         
 related_id           | 510221992                                                        
 event_typ

In [19]:
%%pyspark project.spark.compatibility
dlb_final.printSchema()

root
 |-- record_id: string (nullable = true)
 |-- phone_number_AC_hash: string (nullable = true)
 |-- phone_number_AT_hash: string (nullable = true)
 |-- mno: string (nullable = true)
 |-- imei: string (nullable = true)
 |-- imsi: string (nullable = true)
 |-- date: date (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- correlation_id: string (nullable = false)
 |-- related_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- tac: string (nullable = true)
 |-- oldIMSI: string (nullable = true)
 |-- oldIMEI: string (nullable = true)
 |-- fromADCSnapshot: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- source_name: string (nullable = false)
 |-- source_event_id: long (nullable = true)
 |-- schema_version: integer (nullable = false)
 |-- event_date: date (nullable = true)

Connection: project.spark.compatibility | Run start time: 2026-07-09 02:36:51.717992 | Run duration : 0:00:01.052148s.


In [20]:
%%pyspark project.spark.compatibility
# Persist cleaned silver to a temp parquet location so downstream cells
# (validation, EDA, tests) read from disk instead of re-running the full
# cleaning lineage on every query.
# Mirror of the bronze temp_raw pattern in cell 3-4.
#
# Run this ONCE after any change to the cleaning pipeline. On subsequent
# notebook runs where cleaning is unchanged, you can skip this cell and
# go straight to the validation section (which reads from this path).
dlb_final.write.mode("overwrite").parquet(
    "s3://enstream-lake-silver-dev/temp_raw/device_lookup_batch_cleaned/"
)



Connection: project.spark.compatibility | Run start time: 2026-07-09 02:36:52.774269 | Run duration : 0:10:00.157721s.


# Data Validation

Reload the staged Parquet and assert the business contract on the real data. FAIL counts are intentional KPIs that surface missing/invalid data to report back to carriers.

In [21]:
%%pyspark project.spark.compatibility
# Read the cleaned silver from temp_raw (written by the last cell of the cleaning section).
# This breaks the lineage to the bronze read + cleaning chain, so downstream
# validation / EDA queries are single parquet scans instead of full pipeline re-execution.
dlb_silver = spark.read.parquet(
    "s3://enstream-lake-silver-dev/temp_raw/device_lookup_batch_cleaned/"
)


Connection: project.spark.compatibility | Run start time: 2026-07-09 02:46:52.937959 | Run duration : 0:00:02.313311s.


In [22]:
%%pyspark project.spark.compatibility
# dlb_silver.cache()
# dlb_silver.count()

Connection: project.spark.compatibility | Run start time: 2026-07-09 02:46:55.255410 | Run duration : 0:00:00.049010s.


In [23]:
%%pyspark project.spark.compatibility
dlb_orig_nulls = dlb.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in dlb.columns
])
dlb_orig_nulls.show()


+---+----------+--------------+---+--------+--------+----+---------+--------+----------+------+
| id|change_key|change_key_ats|mno|    imei|    imsi|date|timestamp|   notes|related_id|status|
+---+----------+--------------+---+--------+--------+----+---------+--------+----------+------+
|  0|         0|             0|  0|32785670|67637768|   0|        0|28789505|         0|     0|
+---+----------+--------------+---+--------+--------+----+---------+--------+----------+------+

Connection: project.spark.compatibility | Run start time: 2026-07-09 02:46:55.307924 | Run duration : 0:01:25.489509s.


In [24]:
%%pyspark project.spark.compatibility
# Null counts per column. MISSING_DATA is already converted to null in to_silver
# (treated identically to \N), so a plain isNull() check is enough.
dlb_nulls = dlb_silver.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in dlb_silver.columns
])
dlb_nulls.show()

+---------+--------------------+--------------------+---+--------+--------+----+---------------+--------------+----------+----------+--------+---------+---------+---------------+------------+-----------+---------------+--------------+----------+
|record_id|phone_number_AC_hash|phone_number_AT_hash|mno|    imei|    imsi|date|event_timestamp|correlation_id|related_id|event_type|     tac|  oldIMSI|  oldIMEI|fromADCSnapshot|ingestion_ts|source_name|source_event_id|schema_version|event_date|
+---------+--------------------+--------------------+---+--------+--------+----+---------------+--------------+----------+----------+--------+---------+---------+---------------+------------+-----------+---------------+--------------+----------+
|        0|                   0|            20971673|  0|41043350|75895448|   0|              0|             0|         0|         0|41043407|400619036|362563067|      406627781|           0|          0|              0|             0|         0|
+---------+-----

In [25]:
%%pyspark project.spark.compatibility
# +---+----------+--------------+---+--------+--------+----+---------+--------+----------+------+
# | id|change_key|change_key_ats|mno|    imei|    imsi|date|timestamp|   notes|related_id|status|
# +---+----------+--------------+---+--------+--------+----+---------+--------+----------+------+
# |  0|         0|             0|  0|32785670|67637768|   0|        0|28789505|         0|     0|
# +---+----------+--------------+---+--------+--------+----+---------+--------+----------+------+
# +---------+--------------------+--------------------+---+--------+--------+----+---------------+--------------+----------+----------+--------+---------+---------+---------------+------------+-----------+---------------+--------------+
# |record_id|phone_number_AC_hash|phone_number_AT_hash|mno|    imei|    imsi|date|event_timestamp|correlation_id|related_id|event_type|     tac|  oldIMSI|  oldIMEI|fromADCSnapshot|ingestion_ts|source_name|source_event_id|schema_version|
# +---------+--------------------+--------------------+---+--------+--------+----+---------------+--------------+----------+----------+--------+---------+---------+---------------+------------+-----------+---------------+--------------+
# |        0|                   0|            20971673|  0|41043350|75895448|   0|              0|     412347877|         0|         0|41043407|400619036|362563067|      406627781|           0|          0|      412347877|             0|
# +---------+--------------------+--------------------+---+--------+--------+----+---------------+--------------+----------+----------+--------+---------+---------+---------------+------------+-----------+---------------+--------------+


Connection: project.spark.compatibility | Run start time: 2026-07-09 02:49:28.862937 | Run duration : 0:00:00.051106s.


In [26]:
%%pyspark project.spark.compatibility
dlb_silver.show(10, False)

+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+------+----------------------------------------------------------------+----------------------------------------------------------------+----------+-------------------+----------------------------------------+----------+----------+--------+-------+-------+---------------+-------------------+-----------+---------------+--------------+----------+
|record_id                                                       |phone_number_AC_hash                                            |phone_number_AT_hash                                            |mno   |imei                                                            |imsi                                                            |date      |event_timestamp    |correlation_id                          |related_id|event_type|tac     |oldIMSI|oldIMEI|fromA

In [27]:
%%pyspark project.spark.compatibility
# ============================================================
# Data validation on dlb_silver - business-contract checks
# Sources:
#   - Image 1 (bronze schema): NOT NULL columns
#   - Image 2: mno -> allowed status (event_type) matrix
#   - Helena's note: all nulls are unacceptable; FAIL counts are KPIs to surface
#     missing data with carriers (especially for the future graph database).
# Each check counts "rows that violate the contract"; expected = 0.
# ============================================================

ALLOWED_EVENT_TYPES_BY_MNO = {
    "BELL":   {"IMEI_END", "IMEI_START", "IMSI_END", "IMSI_START"},
    "TELUS":  {"IMEI_END", "IMEI_START", "IMEI_START_A", "IMEI_START_NC",
               "IMSI_END_C", "IMSI_END", "IMSI_START", "IMSI_START_A", "IMSI_START_NC"},
    "ROGERS": {"IMEI_END", "IMEI_START", "IMSI_END", "IMSI_START"},
}
ALL_EVENT_TYPES = set().union(*ALLOWED_EVENT_TYPES_BY_MNO.values())
HEX64 = "(?i)^[0-9a-f]{64}$"

def _check(label, violation_cond):
    """Count rows that violate the contract; print PASS / FAIL with the count."""
    n = dlb_silver.filter(violation_cond).count()
    status = "PASS" if n == 0 else f"FAIL ({n})"
    print(f"{status:<14} {label}")
    return n

print("---- dlb_silver data validation ----")
print(f"total rows: {dlb_silver.count()}\n")

# 1. NOT NULL coverage (Helena: any null is a data-quality issue).
#    source_event_id (= bronze id) and correlation_id (now filled with gen- synthetic)
#    are non-null this revision, so they are NOT skipped -> they get NOT NULL checks.
#    Only the notes-parsed optional fields remain legitimately nullable.
EXPECTED_NULL_COLUMNS = {
    "tac", "oldIMSI", "oldIMEI", "fromADCSnapshot",
}
print("--- NOT NULL coverage ---")
for c in dlb_silver.columns:
    if c in EXPECTED_NULL_COLUMNS:
        continue
    _check(f"{c} IS NOT NULL", F.col(c).isNull())

# 2. mno domain
_check("mno IN (BELL, TELUS, ROGERS)",
       ~F.col("mno").isin(["BELL", "TELUS", "ROGERS"]))

# 3. event_type in the known set
_check(f"event_type IN known set (n={len(ALL_EVENT_TYPES)})",
       ~F.col("event_type").isin(list(ALL_EVENT_TYPES)))

# 4. (mno, event_type) pairs follow Image 2
for mno, allowed in ALLOWED_EVENT_TYPES_BY_MNO.items():
    _check(f"{mno}.event_type subset of allowed list",
           (F.col("mno") == mno) & ~F.col("event_type").isin(list(allowed)))

# 5. Hash format checks (64-char hex)
_check("phone_number_AC_hash matches 64-char hex",
       ~F.col("phone_number_AC_hash").rlike(HEX64))
_check("phone_number_AT_hash (when not null) matches 64-char hex",
       F.col("phone_number_AT_hash").isNotNull() & ~F.col("phone_number_AT_hash").rlike(HEX64))
_check("imei (when not null) matches 64-char hex",
       F.col("imei").isNotNull() & ~F.col("imei").rlike(HEX64))
_check("imsi (when not null) matches 64-char hex",
       F.col("imsi").isNotNull() & ~F.col("imsi").rlike(HEX64))
_check("oldIMEI (when not null) matches 64-char hex",
       F.col("oldIMEI").isNotNull() & ~F.col("oldIMEI").rlike(HEX64))
_check("oldIMSI (when not null) matches 64-char hex",
       F.col("oldIMSI").isNotNull() & ~F.col("oldIMSI").rlike(HEX64))

# 6. tac is 8 digits when not null
_check("tac (when not null) matches 8 digits",
       F.col("tac").isNotNull() & ~F.col("tac").rlike(r"^\d{8}$"))

# 7. related_id (when not null) is numeric
_check("related_id (when not null) is numeric",
       F.col("related_id").isNotNull() & ~F.col("related_id").rlike(r"^\d+$"))

# 7b. fromADCSnapshot is a boolean flag: only 'true'/'false' allowed when not null
#     ('false' left in the domain for future-proofing; current data is 'true' | null).
_check("fromADCSnapshot (when not null) IN ('true','false')",
       F.col("fromADCSnapshot").isNotNull() & ~F.col("fromADCSnapshot").isin("true", "false"))

# 8. record_id is a non-null 64-char hex surrogate (sha2). Positivity no longer
#    applies (it's a hash string, not the bronze bigint id).
_check("record_id IS NOT NULL", F.col("record_id").isNull())
_check("record_id matches 64-char hex", ~F.col("record_id").rlike(HEX64))

# 8b. source_event_id (= bronze id) should be a positive bigint primary key
_check("source_event_id > 0", F.col("source_event_id") <= 0)

# 9. IMEI_START events should carry a non-null imei
_check("IMEI_START => imei IS NOT NULL",
       (F.col("event_type") == "IMEI_START") & F.col("imei").isNull())

# 10. IMSI_START events should carry a non-null imsi
_check("IMSI_START => imsi IS NOT NULL",
       (F.col("event_type") == "IMSI_START") & F.col("imsi").isNull())

# 11. bronze `date` should agree with derived `event_date` (sanity check on source consistency)
_check("date == event_date (source-vs-derived agreement)",
       F.col("date").isNotNull()
       & F.col("event_date").isNotNull()
       & (F.col("date") != F.col("event_date")))


---- dlb_silver data validation ----
total rows: 412347877

--- NOT NULL coverage ---
PASS           record_id IS NOT NULL
PASS           phone_number_AC_hash IS NOT NULL
FAIL (20971673) phone_number_AT_hash IS NOT NULL
PASS           mno IS NOT NULL
FAIL (41043350) imei IS NOT NULL
FAIL (75895448) imsi IS NOT NULL
PASS           date IS NOT NULL
PASS           event_timestamp IS NOT NULL
PASS           correlation_id IS NOT NULL
PASS           related_id IS NOT NULL
PASS           event_type IS NOT NULL
PASS           ingestion_ts IS NOT NULL
PASS           source_name IS NOT NULL
PASS           source_event_id IS NOT NULL
PASS           schema_version IS NOT NULL
PASS           event_date IS NOT NULL
PASS           mno IN (BELL, TELUS, ROGERS)
PASS           event_type IN known set (n=9)
PASS           BELL.event_type subset of allowed list
PASS           TELUS.event_type subset of allowed list
PASS           ROGERS.event_type subset of allowed list
PASS           phone_number_AC_has

# Data Checking (specific problem)

Drill into specific data-quality issues (literal `\N`, non-hex hash IDs, non-8-digit `tac`) with bronze-side views, to take upstream to carriers.

In [28]:
%%pyspark project.spark.compatibility
# Raw bronze rows where change_key_ats is the literal "\N" -- to take to upstream
(dlb
    .filter(F.col("change_key_ats") == "\\N")
    .show(5, truncate=False, vertical=True))

-RECORD 0--------------------------------------------------------------------------
 id             | 3011292                                                          
 change_key     | 30b0be63d9d84a2bfb774887ddb82a4cff6e9dec1d662893b6509e968be46725 
 change_key_ats | \N                                                               
 mno            | ROGERS                                                           
 imei           | 00c87fd00b63cd0cf2b5f6a1a22f225735ba9ef6d6af2ad23f97bfbed713ddcb 
 imsi           | 5e8009e7e7eeda58c69d0c50c9965b17df9e2f2bd1a0840d86b54522de436329 
 date           | 2023-07-07                                                       
 timestamp      | 2023-07-07 08:49:51                                              
 notes          | {"tac":"35486209"}                                               
 related_id     | 2943672                                                          
 status         | IMEI_START                                                

In [29]:
%%pyspark project.spark.compatibility
(dlb
    .filter(F.col("change_key_ats") == "\\N")
    .groupBy("mno", "status")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False))

+------+----------+--------+
|mno   |status    |count   |
+------+----------+--------+
|TELUS |IMEI_START|13225611|
|ROGERS|IMEI_START|4886108 |
|BELL  |IMEI_START|2859954 |
+------+----------+--------+

Connection: project.spark.compatibility | Run start time: 2026-07-09 02:54:45.013417 | Run duration : 0:01:40.736459s.


In [30]:
%%pyspark project.spark.compatibility
# Null breakdown per (mno, event_type) for the 4 hash-id columns.
# Useful for diagnosing WHICH carrier + WHICH event type drives the nulls.

def null_breakdown_by_event(df, col_name):
    return (
        df.groupBy("mno", "event_type")
          .agg(
              F.count("*").alias("total"),
              F.sum(F.col(col_name).isNull().cast("int")).alias(f"{col_name}_null"),
          )
          .withColumn("null_pct", F.round(F.col(f"{col_name}_null") / F.col("total"), 4))
          .orderBy(F.desc(f"{col_name}_null"))
    )

for column_name in ["imei", "imsi"]:
    print(f"=== {column_name} null breakdown by (mno, event_type) ===")
    null_breakdown_by_event(dlb_silver, column_name).show(truncate=False)

=== imei null breakdown by (mno, event_type) ===
+------+-------------+---------+---------+--------+
|mno   |event_type   |total    |imei_null|null_pct|
+------+-------------+---------+---------+--------+
|BELL  |IMEI_END     |32388872 |8606010  |0.2657  |
|TELUS |IMSI_START_A |7896919  |7896919  |1.0     |
|TELUS |IMSI_END_C   |7423013  |7423013  |1.0     |
|ROGERS|IMEI_END     |38723939 |4911174  |0.1268  |
|BELL  |IMSI_END     |4429864  |4429864  |1.0     |
|BELL  |IMSI_START   |4429864  |4429864  |1.0     |
|ROGERS|IMSI_END     |13610060 |3346506  |0.2459  |
|BELL  |IMEI_START   |35248826 |0        |0.0     |
|TELUS |IMSI_START_NC|635729   |0        |0.0     |
|TELUS |IMEI_START_NC|635729   |0        |0.0     |
|ROGERS|IMEI_START   |47602455 |0        |0.0     |
|TELUS |IMEI_START   |199700348|0        |0.0     |
|ROGERS|IMSI_START   |17602599 |0        |0.0     |
|TELUS |IMSI_START   |2019660  |0        |0.0     |
+------+-------------+---------+---------+--------+

=== imsi null 

In [31]:
%%pyspark project.spark.compatibility
# Bronze view: "null" includes actual null + literal "\N" + "MISSING_DATA"
# (these all become null after clean_str in silver)
NULL_LIKE = lambda c: F.col(c).isNull() | (F.col(c) == "\\N") | (F.col(c) == "MISSING_DATA")

print("=== bronze rows where imei is null/\\N/MISSING_DATA ===")
dlb.filter(NULL_LIKE("imei")).show(5, truncate=False, vertical=True)

print("=== bronze rows where imsi is null/\\N/MISSING_DATA ===")
dlb.filter(NULL_LIKE("imsi")).show(5, truncate=False, vertical=True)


=== bronze rows where imei is null/\N/MISSING_DATA ===
-RECORD 0--------------------------------------------------------------------------
 id             | 522785072                                                        
 change_key     | 87c7fcfcc19a393cb9653c0453cc775079c7b0f0d0b3594edcb7d64bdde7607b 
 change_key_ats | 5f47e9049579bd2b41d40a992f9b10cc465581a5ef452b89c3473863d596b725 
 mno            | BELL                                                             
 imei           | NULL                                                             
 imsi           | NULL                                                             
 date           | 2024-01-16                                                       
 timestamp      | 2024-01-16 14:52:20                                              
 notes          | NULL                                                             
 related_id     | 510255272                                                        
 status         | IME

In [32]:
%%pyspark project.spark.compatibility
HEX64  = "(?i)^[0-9a-f]{64}$"
DIGITS = r"^\d+$"

non_hex = dlb_silver.filter(F.col("imei").isNotNull() & ~F.col("imei").rlike(HEX64))
print(f"=== imei non-hex64 total: {non_hex.count():,} ===\n")

print("--- DIGITS bucket (pure digits, top distinct values) ---")
(non_hex.filter(F.col("imei").rlike(DIGITS))
        .groupBy("imei", "mno").count()
        .orderBy(F.desc("count")).show(20, truncate=False))

print("--- OTHER bucket (everything else, top distinct values) ---")
(non_hex.filter(~F.col("imei").rlike(DIGITS))
        .groupBy("imei", "mno").count()
        .orderBy(F.desc("count")).show(20, truncate=False))


=== imei non-hex64 total: 57 ===

--- DIGITS bucket (pure digits, top distinct values) ---
+--------------------------+------+-----+
|imei                      |mno   |count|
+--------------------------+------+-----+
|00115194218327010402003490|ROGERS|1    |
|00115194218328356245002544|ROGERS|1    |
|00115195329568000000000000|ROGERS|1    |
|00112892427454010262001747|ROGERS|1    |
|00116472945804000000000000|ROGERS|1    |
|00115149612678000000000000|ROGERS|1    |
|00112892427466010217009432|ROGERS|1    |
|00119055374524000000000000|ROGERS|1    |
|00116472840498010381008951|ROGERS|1    |
|00112892427494010118720230|ROGERS|1    |
|00119059033068356862007422|ROGERS|1    |
|00117053237464358005003634|ROGERS|1    |
|00119057049061000000000000|ROGERS|1    |
|00115195502836356659000385|ROGERS|1    |
|00112892427476000000000000|ROGERS|1    |
|00115195517583000000000000|ROGERS|1    |
|00114168248616000000000000|ROGERS|1    |
|00116047798886010210007125|ROGERS|1    |
|00112892427479000000000000

In [33]:
%%pyspark project.spark.compatibility
HEX64  = "(?i)^[0-9a-f]{64}$"
DIGITS = r"^\d+$"

non_hex = dlb_silver.filter(F.col("imsi").isNotNull() & ~F.col("imsi").rlike(HEX64))
print(f"=== imsi non-hex64 total: {non_hex.count():,} ===\n")

print("--- DIGITS bucket (pure digits, top distinct values) ---")
(non_hex.filter(F.col("imsi").rlike(DIGITS))
        .groupBy("imsi", "mno").count()
        .orderBy(F.desc("count")).show(20, truncate=False))

print("--- OTHER bucket (everything else, top distinct values) ---")
(non_hex.filter(~F.col("imsi").rlike(DIGITS))
        .groupBy("imsi", "mno").count()
        .orderBy(F.desc("count")).show(20, truncate=False))



=== imsi non-hex64 total: 7,544 ===

--- DIGITS bucket (pure digits, top distinct values) ---
+-----------------------------+----+-----+
|imsi                         |mno |count|
+-----------------------------+----+-----+
|000141600099994160009999     |BELL|4209 |
|00019979020531992690510000531|BELL|335  |
|000199790205309979020530     |BELL|236  |
|00014164577517999990000001117|BELL|233  |
|00014164577529999990000001134|BELL|218  |
|00014164577601999990000001126|BELL|209  |
|00014164577603999990000001130|BELL|201  |
|00014164577528999990000001133|BELL|146  |
|00019979020531999990000001121|BELL|77   |
|00019979020531999990000001111|BELL|77   |
|00019979020531999990000001116|BELL|76   |
|00019979020531999990000001126|BELL|75   |
|00019979020530999990000001130|BELL|69   |
|00019979020530999990000001125|BELL|67   |
|00019979020530999990000001115|BELL|66   |
|00014164577517992690510000531|BELL|53   |
|00014164577523999990000001123|BELL|52   |
|00014164577603999990000001124|BELL|42   |
|00

In [34]:
%%pyspark project.spark.compatibility
# Show up to 5 example rows from BRONZE for each non-hex64 bucket; skip empty buckets.
# Bronze view = source-side column names + original `notes` JSON, useful for upstream.
HEX64  = "(?i)^[0-9a-f]{64}$"
DIGITS = r"^\d+$"

def _show_if_any(label, df, n=5):
    cnt = df.count()
    if cnt == 0:
        return
    print(f"=== {label} (total {cnt:,}, showing up to {n}) ===")
    df.show(n, truncate=False, vertical=True)

# Bronze filter: non-null, non-\N, non-MISSING_DATA, non-hex64
non_hex_imei = dlb.filter(
    F.col("imei").isNotNull()
    & (F.col("imei") != "\\N")
    & (F.col("imei") != "MISSING_DATA")
    & ~F.col("imei").rlike(HEX64)
)
non_hex_imsi = dlb.filter(
    F.col("imsi").isNotNull()
    & (F.col("imsi") != "\\N")
    & (F.col("imsi") != "MISSING_DATA")
    & ~F.col("imsi").rlike(HEX64)
)

_show_if_any("imei DIGITS bucket",
             non_hex_imei.filter(F.col("imei").rlike(DIGITS)))
_show_if_any("imei OTHER bucket",
             non_hex_imei.filter(~F.col("imei").rlike(DIGITS)))
_show_if_any("imsi DIGITS bucket",
             non_hex_imsi.filter(F.col("imsi").rlike(DIGITS)))
_show_if_any("imsi OTHER bucket",
             non_hex_imsi.filter(~F.col("imsi").rlike(DIGITS)))


=== imei DIGITS bucket (total 56, showing up to 5) ===
-RECORD 0--------------------------------------------------------------------------
 id             | 2802641842                                                       
 change_key     | 1371305633a09305a24c9eda794e4783e2d33c6d306359946ca15b856e622277 
 change_key_ats | 0f8969540c143df4c19b1b46f359a041ca96da4b3db11e00216061bc7404152b 
 mno            | ROGERS                                                           
 imei           | 00112892427466010217009432                                       
 imsi           | 4d4525bac3a49e2a78de95669ee39a11558ba863b2c34bfa7ffaeb46f9a02281 
 date           | 2025-06-25                                                       
 timestamp      | 2025-06-25 01:22:55                                              
 notes          | {"oldIMEI":"MISSING_DATA"}                                       
 related_id     | 2731377812                                                       
 status         | IME

In [35]:
%%pyspark project.spark.compatibility
TAC8 = r"^\d{8}$"

non_8digit = dlb_silver.filter(F.col("tac").isNotNull() & ~F.col("tac").rlike(TAC8))
print(f"=== tac non-8-digit total: {non_8digit.count():,} ===\n")
non_8digit.show(20, truncate=False, vertical=True)


=== tac non-8-digit total: 2 ===

-RECORD 0--------------------------------------------------------------------------------
 record_id            | 04888d7c2632282b47e7849637da7a4db38c9635283aa5b38ecb19501e1b9c0f 
 phone_number_AC_hash | 9e3ae5b18021366769249767f7bec0a0c15c68f32c54f669d919370158e1f396 
 phone_number_AT_hash | 7275180453f0a1989e9b5fb5d2829db910dee4392c2ffd60fb4e62e6541ba905 
 mno                  | ROGERS                                                           
 imei                 | 2315332892e25956403f573ece1ac10f6ac3c83d4c67ba849a566a0cf4a04496 
 imsi                 | 9c5e58ecd7d1f34071005b9fb36bf86ee75b0bc04829ef6d9f2180275dc49a20 
 date                 | 2024-01-11                                                       
 event_timestamp      | 2024-01-11 11:52:23                                              
 correlation_id       | gen-04888d7c-2632-282b-47e7-849637da7a4d                         
 related_id           | 480791912                                 

In [36]:
%%pyspark project.spark.compatibility
# Bronze view of the tac=non-8-digit rows; pulls tac out of `notes` JSON on the fly
# so you can see the original notes string + source-side columns for upstream.
TAC8 = r"^\d{8}$"

bronze_tac_rows = (
    dlb
    .withColumn("_tac", F.get_json_object(F.col("notes"), "$.tac"))
    .filter(
        F.col("_tac").isNotNull()
        & ~F.col("_tac").isin("\\N", "", "MISSING_DATA")
        & ~F.col("_tac").rlike(TAC8)
    )
)

print(f"=== bronze rows with non-8-digit tac in notes: {bronze_tac_rows.count():,} ===\n")
bronze_tac_rows.show(20, truncate=False, vertical=True)


=== bronze rows with non-8-digit tac in notes: 2 ===

-RECORD 0--------------------------------------------------------------------------
 id             | 492638262                                                        
 change_key     | 9e3ae5b18021366769249767f7bec0a0c15c68f32c54f669d919370158e1f396 
 change_key_ats | 7275180453f0a1989e9b5fb5d2829db910dee4392c2ffd60fb4e62e6541ba905 
 mno            | ROGERS                                                           
 imei           | 2315332892e25956403f573ece1ac10f6ac3c83d4c67ba849a566a0cf4a04496 
 imsi           | 9c5e58ecd7d1f34071005b9fb36bf86ee75b0bc04829ef6d9f2180275dc49a20 
 date           | 2024-01-11                                                       
 timestamp      | 2024-01-11 11:52:23                                              
 notes          | {"tac":"0", "fromADCSnapshot":true}                              
 related_id     | 480791912                                                        
 status         | IMEI

# Save Data

Publish the cleaned Silver to the partitioned Iceberg table, then verify partitioning.

In [37]:
%%pyspark project.spark.compatibility
def write_iceberg(df, table_name, s3_location, schema_ddl, format_version="3"):
    """Create table if absent + overwrite all partitions covered by df.
    Idempotent for full-reload use case: re-running with the same data is a no-op
    because overwritePartitions atomically replaces each partition's data with df's.

    NOTE: if you change `schema_ddl`, drop the table manually first:
        spark.sql("DROP TABLE IF EXISTS <table_name>")
    """
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            {schema_ddl}
        )
        USING iceberg
        PARTITIONED BY (event_date)
        LOCATION '{s3_location}'
        TBLPROPERTIES ('format-version' = '{format_version}')
    """)
    df.writeTo(table_name).overwritePartitions()


def verify_iceberg(table_name):
    """Run 4 sanity checks + row count."""
    print(f"=== A. SHOW CREATE TABLE ===")
    spark.sql(f"SHOW CREATE TABLE {table_name}").show(truncate=False)
    print(f"=== B. file paths ===")
    spark.sql(f"SELECT file_path FROM {table_name}.files LIMIT 5").show(truncate=False)
    print(f"=== C. file count ===")
    spark.sql(f"SELECT COUNT(*) AS file_count FROM {table_name}.files").show()
    print(f"=== D. partitions ===")
    spark.sql(f"""
        SELECT partition, spec_id, record_count
        FROM {table_name}.partitions ORDER BY partition LIMIT 10
    """).show(truncate=False)
    print(f"row count: {spark.table(table_name).count():,}")


Connection: project.spark.compatibility | Run start time: 2026-07-09 03:16:02.691573 | Run duration : 0:00:01.128186s.


In [38]:
%%pyspark project.spark.compatibility
# Write dlb silver to Iceberg.
# Safe to rerun -- write_iceberg uses CREATE TABLE IF NOT EXISTS + overwritePartitions,
# so re-running with the same data replaces each partition's files (no duplication).
#
# NOTE: record_id changed BIGINT -> STRING this revision (sha2 surrogate). CREATE TABLE
# IF NOT EXISTS will NOT alter an existing table, so writing STRING into the old BIGINT
# column fails on type mismatch. Drop the table ONCE before the first run on the new
# schema, then re-run this cell:
spark.sql("DROP TABLE IF EXISTS trust_score_v1_poc_silver.device_lookup_batch")
write_iceberg(
    df=dlb_silver,
    table_name="trust_score_v1_poc_silver.device_lookup_batch",
    s3_location="s3://enstream-lake-silver-dev/poc/tables/dataset=device_lookup_batch/version=1/",
    schema_ddl="""
        record_id STRING,
        phone_number_AC_hash STRING,
        phone_number_AT_hash STRING,
        mno STRING,
        imei STRING,
        imsi STRING,
        date DATE,
        event_timestamp TIMESTAMP,
        correlation_id STRING,
        related_id STRING,
        event_type STRING,
        tac STRING,
        oldIMSI STRING,
        oldIMEI STRING,
        fromADCSnapshot STRING,
        ingestion_ts TIMESTAMP,
        source_name STRING,
        source_event_id BIGINT,
        schema_version INT,
        event_date DATE
    """,
)

verify_iceberg("trust_score_v1_poc_silver.device_lookup_batch")


Py4JJavaError:An error occurred while calling o1408.overwritePartitions.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1052 in stage 275.0 failed 4 times, most recent failure: Lost task 1052.3 in stage 275.0 (TID 64742) (10.0.2.199 executor 2): java.io.FileNotFoundException: /tmp/blockmgr-9a1f837a-dfe4-4a3a-a6dc-3ac0c08ebcce/1e/temp_shuffle_ab0408c1-953a-4b1c-befd-2b99b7f55d42 (No space left on device)
	at java.base/java.io.FileOutputStream.open0(Native Method)
	at java.base/java.io.FileOutputStream.open(FileOutputStream.java:293)
	at java.base/java.io.FileOutputStream.<init>(FileOutputStream.java:235)
	at org.apache.spark.storage.DiskBlockObjectWriter.initialize(DiskBlockObjectWriter.scala:151)
	at org.apache.spark.storage.DiskBlockObjectWriter.open(DiskBlockObjectWriter.scala:171)
	at org.apache.spark.storage.DiskBlockObjectWriter.write(DiskBlockObjectWriter.scala:358)
	at org.apache.spark.serializer.SerializedRecordWriter$.copyBytes(SerializedRecordWriter

Connection: project.spark.compatibility | Run start time: 2026-07-09 03:16:03.823988 | Run duration : 0:02:13.057473s.


ExecutionException: Code execution failed

In [ ]:
%%pyspark project.spark.compatibility
# Inspect the partition folders Iceberg actually created in S3.
s3 = boto3.client('s3')
resp = s3.list_objects_v2(
    Bucket='enstream-lake-silver-dev',
    Prefix='poc/tables/dataset=device_lookup_batch/version=1/data/',
    Delimiter='/',         # key bit: list folders only, do not recurse
    MaxKeys=30
)

# the folder names S3 actually returns
for p in resp.get('CommonPrefixes', []):
    print(p['Prefix'])
